# SIC8 | 01 — Historical Source Profiling

**Role 2: Spark & Fraud Detection · Databricks + Amazon S3**

**هدفنا:** نتأكد إننا بنقرأ **الـHistorical Transactions CSV الأصلية فقط**، نفهم الـ15 Columns، ونقيس Data Quality قبل أي Cleaning أو Silver Mapping.

**مهم قبل الـRun**
- الـSource في `raw/baseline/` **Read-only**. الـNotebook دي لا تكتب على S3 أو Delta.
- `SAMPLE` للتجربة السريعة (أول صفوف موجودة، **مش عينة عشوائية ممثلة**). بعدها `FULL` لتقييم كل الداتا.
- كل عملية كبيرة متوضحة إنها **Spark Action**، وممكن تاخد وقت وتستهلك Credits.
- `Is Fraud?` هنا للـ**Profiling** فقط. **ممنوع** استخدامه كـDetection Feature.
- الـHistorical Dataset مفيهاش `transaction_id` و`account_id` وGPS دقيقة. **مش هنخترعهم في الـProfiling.**

**Execution guide:** امشِ على الخلايا بالترتيب. اقرأ الملاحظات اللي فوق كل خلية، وشغّل Full Profile مرة واحدة بعد نجاح الـSample.

## 1 — Configuration (عدّل الإعدادات من هنا فقط)

أسماء الـVolume دي مبنية على الاتصال اللي نجح عندك بالفعل. لو غيرت أسماء الـCatalog/Schema عدّل `RAW_PATH` فقط. **متقرأش المجلد كله بـ`.csv(RAW_PATH)`** لأنه فيه Users وCards وغيرهم، وكل ملف Schema مختلفة.

In [0]:
# 01: Project settings
RAW_PATH = "/Volumes/workspace/sic8_sources/historical_raw"
TRANSACTIONS_PATH = f"{RAW_PATH}/credit_card_transactions-ibm_v2.csv"

# Start small. Change to "FULL" only after SAMPLE runs successfully.
PROFILE_MODE = "SAMPLE"          # "SAMPLE" or "FULL"
SAMPLE_ROWS = 5000

# Previously reported values - NOT verified until we run FULL profiling.
PREVIOUS_TRANSACTION_COUNT = 24_386_900
PREVIOUS_FRAUD_COUNT = 29_757

EXPECTED_COLUMNS = [
    "User", "Card", "Year", "Month", "Day", "Time", "Amount",
    "Use Chip", "Merchant Name", "Merchant City", "Merchant State",
    "Zip", "MCC", "Errors?", "Is Fraud?"
]

print("Profile mode:", PROFILE_MODE)
print("Transaction file:", TRANSACTIONS_PATH)

Profile mode: SAMPLE
Transaction file: /Volumes/workspace/sic8_sources/historical_raw/credit_card_transactions-ibm_v2.csv


## 2 — Environment / Compute & connection

**الخلية دي خفيفة:** تجيب Spark Version وWorkspace context، وتعرض الـFiles. **نوع الـCompute الحقيقي هتشوفه في الـNotebook top bar → Compute selector**:
- **Serverless:** شوف `See performance` تحت الخلية بعد التنفيذ ثم `Query profile` (غالبًا مفيش Spark UI تقليدية).
- **Classic:** من **Compute** افتح الـCluster وشوف Workers / Driver / Spark UI / Auto termination.

> `spark.version` يوضح نسخة Spark، **مش** CPU/RAM/عدد الـWorkers. متخمنش القوة من اسم الـNotebook.

In [0]:
# 02: Lightweight environment check
print("Spark version:", spark.version)

# Catalog and schema are Databricks SQL context, not machine size.
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)

# S3-backed external volume is already connected.
for item in dbutils.fs.ls(RAW_PATH):
    print(item.name, "| bytes:", item.size)

print("Compute details: check the Compute selector at the top of this notebook.")

Spark version: 4.2.0
+---------+-------+
|catalog  |schema |
+---------+-------+
|workspace|default|
+---------+-------+

city_reference.csv | bytes: 104813
credit_card_transactions-ibm_v2.csv | bytes: 2350744057
merchant_category_codes.csv | bytes: 52651
sd254_cards.csv | bytes: 487120
sd254_users.csv | bytes: 224394
Compute details: check the Compute selector at the top of this notebook.


## 3 — Read ONE CSV (without automatic transformations)

CSV Raw Columns هتفضل `string` عمدًا. ده أحسن من `inferSchema` اللي ممكن يحول ZIP أو MCC لنوع مش مناسب.

`enforceSchema=false` يساعد Spark يتحقق إن الـHeader مطابق للـColumns بدل ما يطبّق أسماء الأعمدة بصمت على ملف مختلف. `mode=PERMISSIVE` لا يثبت وحده أن **كل CSV row structurally valid**؛ Validation أكثر صرامة هتحتاج فحص مستقل بعدين.

In [0]:
# 03: Read the historical transactions file only
from pyspark.sql import functions as F
import time

read_start = time.perf_counter()

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("enforceSchema", "false")
    .option("mode", "PERMISSIVE")
    .csv(TRANSACTIONS_PATH)
)

print("DataFrame setup (not full scan):", round(time.perf_counter() - read_start, 2), "sec")
print("Columns found:", len(df.columns))
print("Column names match expected:", df.columns == EXPECTED_COLUMNS)

if df.columns != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected columns! Found: {df.columns}")

df.printSchema()

DataFrame setup (not full scan): 0.0 sec
Columns found: 15
Column names match expected: True
root
 |-- User: string (nullable = true)
 |-- Card: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Day: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Amount: string (nullable = true)
 |-- Use Chip: string (nullable = true)
 |-- Merchant Name: string (nullable = true)
 |-- Merchant City: string (nullable = true)
 |-- Merchant State: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- MCC: string (nullable = true)
 |-- Errors?: string (nullable = true)
 |-- Is Fraud?: string (nullable = true)



## 4 — First action / timing test

دي أول مرة نشغّل **Spark Action** على ملف S3. هنقيس `display` بدون عرض معلومات حساسة من Users/Cards. أول مرة ممكن تكون أبطأ بسبب Session start / planning / S3.

> مهم: `limit(5)` **لا يقيس سرعة قراءة 2.35GB كلها**. ده Smoke Test للـSource والـCompute. بعد التنفيذ جرّب `See performance` لمعرفة أين ضاع الوقت.

In [0]:
# 04: Minimal preview and wall-clock time
preview_columns = [
    "User", "Card", "Year", "Month", "Day", "Time",
    "Amount", "Use Chip", "MCC", "Is Fraud?"
]

start = time.perf_counter()
display(df.select(*preview_columns).limit(5))
print("First preview:", round(time.perf_counter() - start, 2), "sec")

start = time.perf_counter()
second_preview = df.select("User", "Card", "Amount").limit(5).collect()
print("Second small read:", round(time.perf_counter() - start, 2), "sec")
print("Rows returned:", len(second_preview))

User,Card,Year,Month,Day,Time,Amount,Use Chip,MCC,Is Fraud?
0,0,2002,9,1,06:21,$134.09,Swipe Transaction,5300,No
0,0,2002,9,1,06:42,$38.48,Swipe Transaction,5411,No
0,0,2002,9,2,06:22,$120.34,Swipe Transaction,5411,No
0,0,2002,9,2,17:45,$128.95,Swipe Transaction,5651,No
0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5912,No


First preview: 1.53 sec
Second small read: 0.7 sec
Rows returned: 5


## 5 — Define the scope: SAMPLE first, then FULL

في `SAMPLE` بنفحص **أول 5,000 صف حسب خطة القراءة** فقط. النتائج دي **مش نسب حقيقية عن الـDataset**. بعد نجاحها، ارجع لخلية 1، غيّر `PROFILE_MODE` إلى `FULL`، وشغّل من الخلية دي وما بعدها.

**ليه؟** علشان نفهم الكود ونقيس النتائج قبل عمل Full Scan مكلفة.

In [0]:
# 05: Choose scope
if PROFILE_MODE not in ("SAMPLE", "FULL"):
    raise ValueError("PROFILE_MODE must be SAMPLE or FULL")

if PROFILE_MODE == "SAMPLE":
    working_df = df.limit(SAMPLE_ROWS)
    print(f"SAMPLE mode: at most {SAMPLE_ROWS:,} first rows (NOT representative)")
else:
    working_df = df
    print("FULL mode: scanning the historical transaction file")

SAMPLE mode: at most 5,000 first rows (NOT representative)


## 6 — Add temporary *profiling-only* expressions

لسه **مش بننضف أو نكتب Silver**. بنجهز أعمدة مؤقتة علشان نعرف إيه اللي ممكن يفشل عند التحويل.

- `$134.09` أو `$-99.00` → `DECIMAL(18,2)` بـ`try_cast` (القيمة الأصلية محفوظة).
- `Year/Month/Day/Time` → Timestamp **naive** للتأكد من parse؛ الـTimezone الأصلية **غير مثبتة**، فممنوع نفترض UTC.
- `Zip` يفضل String؛ `Is Fraud?` تفضل Label منفصلة.
- `try_cast` و`try_to_timestamp` بتسيب Invalid Values `NULL` بدل Job Error.

In [0]:
# 06: Temporary validation columns; no overwriting of raw fields
profile_df = (
    working_df
    .withColumn(
        "_amount_decimal",
        F.expr("try_cast(regexp_replace(trim(`Amount`), '[$,]', '') AS DECIMAL(18,2))")
    )
    .withColumn(
        "_parsed_event_time",
        F.expr("""
            try_to_timestamp(
                concat(
                    lpad(trim(`Year`), 4, '0'), '-',
                    lpad(trim(`Month`), 2, '0'), '-',
                    lpad(trim(`Day`), 2, '0'), ' ', trim(`Time`)
                ),
                'yyyy-MM-dd HH:mm'
            )
        """)
    )
)

print("Added temporary columns: _amount_decimal, _parsed_event_time")
print("No data was written to S3 / Delta.")

Added temporary columns: _amount_decimal, _parsed_event_time
No data was written to S3 / Delta.


## 7 — Data quality profile in ONE scan

هنحسب **صف واحد** فيه أهم إحصائيات الجودة. الـAggregations كلها في `agg` واحدة لتجنب عشرات الـFull Scans.

**فرق مهم:** `null` و`empty string` مختلفين. عمود `Errors?` أو `Merchant State` ممكن يكون فاضي بشكل مشروع؛ هنقيسهم **لكن مش هنملأهم عشوائيًا**.

> Full mode: دي Heavy Spark Action على ملف حوالي 2.35GB؛ هتبدأ قراءة فعلية وقد تستغرق وقت. تابع `See performance`.

In [0]:
# 07: Main profiling metrics (one Spark action)
from pyspark.sql.functions import col

def missing_value(c):
    return col(c).isNull() | (F.trim(col(c)) == "")

def nrows(condition, name):
    return F.sum(F.when(condition, 1).otherwise(0)).cast("long").alias(name)

metrics = [F.count("*").alias("total_rows")]

# Source-level missingness. These are measurements, not automatic cleaning rules.
for c in EXPECTED_COLUMNS:
    metrics.append(nrows(missing_value(c), f"missing__{c}"))

# Important parsing and contract checks.
metrics += [
    nrows(~missing_value("Amount") & col("_amount_decimal").isNull(), "invalid_amount"),
    nrows(col("_amount_decimal") < 0, "negative_amount"),
    nrows(col("_amount_decimal") == 0, "zero_amount"),
    nrows(~missing_value("Time") & col("_parsed_event_time").isNull(), "invalid_event_time"),
    nrows(~missing_value("User") & ~F.trim(col("User")).rlike(r"^[0-9]+$"), "invalid_user_id"),
    nrows(~missing_value("Card") & ~F.trim(col("Card")).rlike(r"^[0-9]+$"), "invalid_card_index"),
    nrows(~missing_value("MCC") & ~F.trim(col("MCC")).rlike(r"^[0-9]{4}$"), "invalid_mcc_format"),
    nrows(~missing_value("Is Fraud?") & ~F.trim(col("Is Fraud?")).isin("Yes", "No"), "unknown_fraud_label"),
    F.min("_amount_decimal").alias("min_amount"),
    F.max("_amount_decimal").alias("max_amount"),
    F.min("_parsed_event_time").alias("min_event_time_local_unknown_tz"),
    F.max("_parsed_event_time").alias("max_event_time_local_unknown_tz"),
]

start = time.perf_counter()
result_row = profile_df.agg(*metrics).first()   # ACTION: runs the profiler
profile_seconds = round(time.perf_counter() - start, 2)
profile_result = result_row.asDict()
print("Mode:", PROFILE_MODE, "| Profiling execution:", profile_seconds, "seconds")
print("Rows checked:", f"{profile_result['total_rows']:,}")

# Small output table; safe to display (only aggregate metrics, not raw PII).
summary_rows = [(name, str(value)) for name, value in profile_result.items()]
display(spark.createDataFrame(summary_rows, ["metric", "value"]))

Mode: SAMPLE | Profiling execution: 5.63 seconds
Rows checked: 5,000


metric,value
total_rows,5000
missing__User,0
missing__Card,0
missing__Year,0
missing__Month,0
missing__Day,0
missing__Time,0
missing__Amount,0
missing__Use Chip,0
missing__Merchant Name,0


## 8 — Distribution (Fraud label and payment method)

دي **Separate Spark Actions**. فيها Data Quality Signals مهمة، بس النتائج من `SAMPLE` مش ممثلة للمجتمع الكامل. `Is Fraud?` هنستخدمه هنا للتقييم الإحصائي، مش كـInput للـRules.

In [0]:
# 08: Small group-bys. On FULL these re-scan the source.
print("Mode:", PROFILE_MODE)

print("Fraud ground-truth distribution:")
display(working_df.groupBy("Is Fraud?").count().orderBy(F.desc("count")))

print("Payment method (Use Chip) distribution:")
display(working_df.groupBy("Use Chip").count().orderBy(F.desc("count")))

Mode: SAMPLE
Fraud ground-truth distribution:


Is Fraud?,count
No,4987
Yes,13


Payment method (Use Chip) distribution:


Use Chip,count
Swipe Transaction,3294
Chip Transaction,887
Online Transaction,819


## 9 — Interpret the results; do NOT clean yet

**قرارات لازم تتاخد قبل Silver:**

1. `User` + `Card` = **Composite Card Identity**؛ رقم `Card` لوحده مش Global.
2. Source **does not provide durable transaction ID**. نستخرج Stable ID من Snapshot + File + Deterministic Row Ordinal (implementation/test later) بدل `monotonically_increasing_id()`.
3. `Is Fraud?` = **Ground Truth**؛ نفصله عن Detector Input.
4. `Errors?`, `Merchant State`, `Zip` الفاضيين **مش تلقائيًا Invalid**. ممنوع نملاهم `No Error` / `ONLINE` / `00000`.
5. Location from reference tables = approximate/unknown; **no fake exact GPS**, ولا `(0,0)`.
6. Currency: Historical Amounts described as USD in source assessment, but **no currency column in this CSV**. وثّق الـSource Contract قبل مقارنة Live EUR.
7. `MCC` و`Zip` أكواد، مش كميات حسابية.
8. **User/Card reference CSVs contain sensitive fields.** متعرضش Card Number أو CVV في الـNotebook أو Logs.

الـNotebook دي ما بتستنتجش إن كل الـCSV Rows structurally correct من فحص Header وعدد الصفوف بس. لو اتكشف اختلاف Counts أو Parse Failures، هنضيف **Strict CSV Validation/Quarantine** ضمن Implementation.

## 10 — Decision / scope report (no write)

دي Summary بسيطة تقدر تنقلها للشات أو تحطها في Team Notes. بتفصل **Verified in THIS run** عن **previous report**.

> بعد ما تشغّل FULL، ابعتلي output هذه الخلية وMain profiling table. نراجع anomalies ونبني `02_Historical_Canonical_Mapping` بناءً على نتائج فعلية.

In [0]:
# 10: Final summary and reported-baseline comparison
rows = profile_result["total_rows"]

print("SIC8 - Historical Transactions / Source Profiling")
print("Mode:", PROFILE_MODE)
print("Current file:", TRANSACTIONS_PATH)
print("Expected source columns:", len(EXPECTED_COLUMNS), "| Found:", len(df.columns))
print("Rows checked in this run:", f"{rows:,}")
print("Missing Amount:", profile_result["missing__Amount"])
print("Invalid Amount:", profile_result["invalid_amount"])
print("Invalid Timestamp:", profile_result["invalid_event_time"])
print("Negative Amounts:", profile_result["negative_amount"])
print("Unknown Fraud Labels:", profile_result["unknown_fraud_label"])
print("Profiling execution (seconds):", profile_seconds)

if PROFILE_MODE == "FULL":
    print("Compared with PREVIOUS report count:", PREVIOUS_TRANSACTION_COUNT)
    print("Count difference:", rows - PREVIOUS_TRANSACTION_COUNT)
    if rows != PREVIOUS_TRANSACTION_COUNT:
        print("REVIEW: source snapshot/count mismatch; do not assume ingestion failure without checking version.")
else:
    print("SAMPLE only: do NOT compare with whole-dataset totals / fraud rate.")

print("Status: profiling complete for current scope. No cleaning or Delta writes performed.")

SIC8 - Historical Transactions / Source Profiling
Mode: SAMPLE
Current file: /Volumes/workspace/sic8_sources/historical_raw/credit_card_transactions-ibm_v2.csv
Expected source columns: 15 | Found: 15
Rows checked in this run: 5,000
Missing Amount: 0
Invalid Amount: 0
Invalid Timestamp: 0
Negative Amounts: 174
Unknown Fraud Labels: 0
Profiling execution (seconds): 5.63
SAMPLE only: do NOT compare with whole-dataset totals / fraud rate.
Status: profiling complete for current scope. No cleaning or Delta writes performed.


## 11 — Performance / Compute quick guide

**الـ30–50 ثانية ممكن تيجي من:** Cold start، Read/Listing Metadata من S3، Query Planning، أو تحميل أول Block. الـCPU/RAM مش السبب الوحيد.

**إزاي أعرف الـCompute اللي شغالة؟**
- **Notebook top bar → Compute dropdown:** Serverless أو اسم Classic cluster.
- **Serverless:** افتح `See performance` بعد أي Action → `Query profile` علشان Execution Time وScan/Stages. الـSpark UI التقليدية غالبًا غير متاحة.
- **Classic:** **Compute → your cluster**، هتلاقي Worker type / Worker count / Autoscale / Driver و`Spark UI`.
- **Cost:** شغّل SAMPLE أولًا وFULL مرة بعد التحقق. ما تعملش `.count()` و`.distinct().count()` لكل Column في Loop (كل Action بتعيد القراءة). `display(5)` مش Benchmark للـFull Dataset.
- **Source:** `raw/baseline` Read-only. No caching 2.35GB in notebook by default, no new clusters until we measure.

**Next notebook after actual results:** `02_Historical_Canonical_Mapping` → explicit data types + stable IDs strategy + source lineage + quality gates + unit tests. بعدها فقط `03_Silver_Delta`.